In [ ]:
import sys, os, glob, subprocess

WHEELS_DIR = '/kaggle/input/datasets/uday192007/offline-wheels'

if not os.path.isdir(WHEELS_DIR):
    raise FileNotFoundError(
        f'Wheels dataset not found at {WHEELS_DIR}.\n'
        'Add the "offline-wheels" dataset to this notebook.'
    )

whl_files = glob.glob(os.path.join(WHEELS_DIR, '**', '*.whl'), recursive=True)
if not whl_files:
    raise FileNotFoundError(f'No .whl files found under {WHEELS_DIR}')

folders = list(set(os.path.dirname(p) for p in whl_files))
find_links_args = []
for folder in folders:
    find_links_args += ['--find-links', folder]

print(f'Found {len(whl_files)} wheels. Installing offline...')
ret = subprocess.call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index']
    + find_links_args
    + ['rasterio', 'albumentations']
)
if ret != 0:
    print('[WARN] find-links install returned non-zero. Trying --no-deps direct install...')
    subprocess.call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps'] + whl_files
    )

import rasterio
print(f'rasterio {rasterio.__version__} ready.')


In [ ]:
import os, math, random, glob, json, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import rasterio

# GPU
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {num_gpus}')
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  VRAM={p.total_memory/1e9:.1f}GB  Compute={p.major}.{p.minor}')

# RTX 6000 Pro / Ada / Ampere optimisations
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark     = True
    torch.backends.cudnn.deterministic = False
    torch.set_float32_matmul_precision('high')   # TF32: ~2-3x matmul on Ampere+
    print('cuDNN benchmark ON  |  TF32 matmul ON')


In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────
def find_data_dir():
    primary = '/kaggle/input/datasets/aaryamanbisht/satdat/training_data'
    if os.path.isdir(primary):
        return primary
    # Fallback: search anywhere under /kaggle/input for clear+cloudy+sar
    for root, dirs, _ in os.walk('/kaggle/input'):
        if {'clear', 'cloudy', 'sar'}.issubset(set(dirs)):
            return root
    raise FileNotFoundError(
        'Cannot find training_data with clear/cloudy/sar under /kaggle/input.\n'
        'Make sure the "satdat" dataset is added to this notebook.'
    )

DATA_DIR       = find_data_dir()
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Hardware ─────────────────────────────────────────────────────────────
MIXED_PRECISION = 'none'   # BF16 = native Ampere/Ada, zero overhead
COMPILE_MODEL   = True     # torch.compile saves ~15% wall-clock per epoch

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS      = 1000
BATCH_SIZE  = 20           # 20 as in OS-MambaDiff paper
LR          = 5e-5
LR_MIN      = 1e-8
NUM_WORKERS = 4            # Kaggle provides 4 CPU cores

# ── Diffusion ────────────────────────────────────────────────────────────
NUM_TIMESTEPS  = 2000
NOISE_SCHEDULE = 'sigmoid'

# ── Model (Faithful paper specs) ─────────────────────────────────────────
MODEL_CHANNELS = 128
MAMBA_CHANNELS = 128
TIME_EMB_DIM   = 512

# ── Loss Weights ─────────────────────────────────────────────────────────
# SAM is NOT in the paper's loss formulation
LAMBDA_MSE   = 0.5
LAMBDA_SSIM  = 0.5
LAMBDA_GATE  = 0.005
LAMBDA_CLOUD = 0.3
CLOUD_MARGIN = 0.2

# ── EMA ──────────────────────────────────────────────────────────────────
EMA_DECAY = 0.999
GRAD_CLIP = 1.0

# ── Validation & Checkpointing ───────────────────────────────────────────
VAL_INTERVAL    = 10    # Validate every epoch to get regular metrics
SAVE_INTERVAL   = 9999 # Disabled periodic checkpoints to save VRAM and disk space
VIS_INTERVAL    = 10   # Visualize sample image grids inline every 10 epochs
VIS_SAMPLES     = 4    # Number of validation samples to display in matplotlib grid
DDIM_STEPS      = 50   # DDIM reverse steps
DDIM_ETA        = 0.0  # 0 = deterministic
MAX_VAL_BATCHES = 0    # 0 = use ALL validation samples

# ── Early Stopping ───────────────────────────────────────────────────────
ES_PATIENCE  = 3
ES_MIN_DELTA = 0.01    # min PSNR improvement in dB

print('─' * 60)
print(f'  DATA_DIR       : {DATA_DIR}')
print(f'  CHECKPOINT_DIR : {CHECKPOINT_DIR}')
print(f'  Mixed precision: {MIXED_PRECISION}')
print(f'  torch.compile  : {COMPILE_MODEL}')
print(f'  Epochs={EPOCHS}  Batch={BATCH_SIZE}  LR={LR}')
print(f'  Val every {VAL_INTERVAL} epoch(s)')
print('─' * 60)


In [ ]:
def normalize_optical(arr, norm_max=500.0):
    """Fixed global scale. LISS-IV DN ~28-715 -> clip(arr/500) -> [0,1]."""
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return np.clip(arr / norm_max, 0.0, 1.0)

def normalize_sar(arr, eps=1e-8):
    """dB -> linear power (10^(x/10)) -> per-image min-max [0,1]."""
    arr    = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    arr    = np.clip(arr, -50.0, 20.0)
    linear = np.power(10.0, arr / 10.0)
    lo, hi = linear.min(), linear.max()
    return (linear - lo) / (hi - lo + eps)

class LISS4Dataset(Dataset):
    """
    LISS-IV Cloud Removal Dataset.
    base_dir/
        clear/   -- cloud-free LISS-IV targets  (.tif, 3 bands, uint16)
        cloudy/  -- cloudy LISS-IV inputs        (.tif, 3 bands, uint16)
        sar/     -- Sentinel-1 SAR (VV/VH)       (.tif, 2 bands, dB float64)
    """
    def __init__(self, base_dir, split='train', split_ratio=0.9, seed=42):
        self.base_dir, self.split = base_dir, split
        sets = {}
        for d in ['clear', 'cloudy', 'sar']:
            dp = os.path.join(base_dir, d)
            if not os.path.isdir(dp):
                raise FileNotFoundError(f'Missing directory: {dp}')
            sets[d] = set(os.path.basename(f)
                          for f in glob.glob(os.path.join(dp, '*.tif')))
        common = sorted(set.intersection(*sets.values()))
        if not common:
            raise FileNotFoundError(f'No matched .tif samples found in {base_dir}')
        random.seed(seed)
        random.shuffle(common)
        n = int(len(common) * split_ratio)
        self.files = common[:n] if split == 'train' else common[n:]
        print(f'LISS4Dataset [{split}]: {len(self.files)} samples')

    def __len__(self): return len(self.files)

    def _read(self, path):
        with rasterio.open(path) as src:
            d = src.read().astype(np.float32)
        return np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)

    def __getitem__(self, idx):
        fn     = self.files[idx]
        clear  = normalize_optical(self._read(os.path.join(self.base_dir, 'clear',  fn)))
        cloudy = normalize_optical(self._read(os.path.join(self.base_dir, 'cloudy', fn)))
        sar    = normalize_sar    (self._read(os.path.join(self.base_dir, 'sar',    fn)))
        clear  = torch.tensor(clear,  dtype=torch.float32)
        cloudy = torch.tensor(cloudy, dtype=torch.float32)
        sar    = torch.tensor(sar,    dtype=torch.float32)
        if self.split == 'train':
            if random.random() > 0.5:
                clear  = torch.flip(clear,  [2])
                cloudy = torch.flip(cloudy, [2])
                sar    = torch.flip(sar,    [2])
            if random.random() > 0.5:
                clear  = torch.flip(clear,  [1])
                cloudy = torch.flip(cloudy, [1])
                sar    = torch.flip(sar,    [1])
        return {'clear': clear, 'cloudy': cloudy, 'sar': sar}


In [ ]:
def compute_psnr(gt, pred, max_val=1.0):
    """10·log10(MAX²/MSE). Valid ONLY after full DDIM sampling."""
    mse = F.mse_loss(pred.float(), gt.float())
    if mse.item() == 0:
        return torch.tensor(100.0)
    return 10.0 * torch.log10(torch.tensor(max_val**2) / mse)

def compute_ssim(gt, pred, win=11, C1=0.01**2, C2=0.03**2):
    """Structural similarity (mean over batch & channels)."""
    gt, pred = gt.float(), pred.float()
    p = win // 2
    mu1  = F.avg_pool2d(gt,      win, stride=1, padding=p)
    mu2  = F.avg_pool2d(pred,    win, stride=1, padding=p)
    s1   = F.avg_pool2d(gt*gt,   win, stride=1, padding=p) - mu1**2
    s2   = F.avg_pool2d(pred**2, win, stride=1, padding=p) - mu2**2
    s12  = F.avg_pool2d(gt*pred, win, stride=1, padding=p) - mu1*mu2
    num  = (2*mu1*mu2 + C1) * (2*s12 + C2)
    den  = (mu1**2 + mu2**2 + C1) * (s1 + s2 + C2)
    return (num / den).mean()

def compute_sam(gt, pred, eps=1e-8):
    """Spectral Angle Mapper (radians, mean over batch & spatial)."""
    gt, pred = gt.float(), pred.float()
    dot  = (gt * pred).sum(dim=1)
    n1   = gt.norm(dim=1).clamp(min=eps)
    n2   = pred.norm(dim=1).clamp(min=eps)
    return torch.acos((dot / (n1*n2)).clamp(-1.0 + 1e-6, 1.0 - 1e-6)).mean()

print('PSNR, SSIM, SAM defined.')


In [ ]:
class DiffusionScheduler:
    """Sigmoid noise schedule + DDIM deterministic reverse sampler."""

    def __init__(self, num_timesteps=2000, schedule='sigmoid'):
        T = num_timesteps
        self.T = T
        if schedule == 'sigmoid':
            steps   = torch.linspace(0, T, T+1)
            t_norm  = steps / T
            vs = torch.sigmoid(torch.tensor(-6.0))
            ve = torch.sigmoid(torch.tensor( 6.0))
            ac = (ve - torch.sigmoid(t_norm * 12.0 - 6.0)) / (ve - vs)
            ac = ac / ac[0]
            betas = torch.clamp(1.0 - ac[1:] / ac[:-1], 0.0, 0.999)
        else:
            betas = torch.linspace(1e-4, 0.02, T)

        alphas = 1.0 - betas
        ac     = torch.cumprod(alphas, 0)

        self.alphas_cumprod      = ac
        self.sqrt_ac             = torch.sqrt(ac)
        self.sqrt_one_minus_ac   = torch.sqrt(1.0 - ac)

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        tc    = t.cpu()
        alpha = self.sqrt_ac[tc].to(x0.device)[:, None, None, None]
        sigma = self.sqrt_one_minus_ac[tc].to(x0.device)[:, None, None, None]
        return alpha * x0 + sigma * noise

    @torch.no_grad()
    def sample_ddim(self, model, cloudy, sar, steps=50, eta=0.0):
        model.eval()
        idxs = torch.linspace(0, self.T-1, steps, dtype=torch.long)
        ts   = list(reversed(idxs.tolist()))
        xt   = torch.randn_like(cloudy)
        pred = None
        for i, tv in enumerate(ts):
            t      = torch.full((cloudy.size(0),), tv, device=cloudy.device, dtype=torch.long)
            p0, _  = model(xt, t, cloudy, sar)
            p0     = torch.clamp(p0, 0.0, 1.0)
            pred   = p0
            ac_t   = self.alphas_cumprod[int(tv)].to(xt.device)
            eps    = (xt - torch.sqrt(ac_t)*p0) / (torch.sqrt(1.0-ac_t)+1e-8)
            if i+1 < len(ts):
                ac_prev = self.alphas_cumprod[int(ts[i+1])].to(xt.device)
            else:
                ac_prev = torch.tensor(1.0, device=xt.device)
            sigma  = eta * torch.sqrt((1-ac_prev)/(1-ac_t+1e-8)*(1-ac_t/(ac_prev+1e-8)))
            dir_xt = torch.sqrt(torch.clamp(1-ac_prev-sigma**2, min=0.0)) * eps
            noise  = torch.randn_like(xt) if eta > 0 else torch.zeros_like(xt)
            xt     = torch.sqrt(ac_prev) * p0 + dir_xt + sigma * noise
        return torch.clamp(pred if pred is not None else xt, 0.0, 1.0)

print('DiffusionScheduler ready.')


In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal time-step embeddings (DDPM / Eq.9 in OS-MambaDiff paper)."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device   = time.device
        half_dim = self.dim // 2
        emb      = math.log(10000) / (half_dim - 1)
        emb      = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb      = time[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class SS2D(nn.Module):
    """
    2-D Selective Scan block (SS2D) from Vision Mamba / VMamba.
    Approximated with 4-directional bidirectional depthwise convolutions.
    Includes gated input projection (Mamba-style expand-gate-contract).
    """
    def __init__(self, channels):
        super().__init__()
        self.in_proj = nn.Conv2d(channels, channels * 2, 1)
        self.act     = nn.SiLU()
        C = channels
        self.dw_h7  = nn.Conv2d(C, C, (1,  7), padding=(0,  3), groups=C)
        self.dw_v7  = nn.Conv2d(C, C, (7,  1), padding=(3,  0), groups=C)
        self.dw_h15 = nn.Conv2d(C, C, (1, 15), padding=(0,  7), groups=C)
        self.dw_v15 = nn.Conv2d(C, C, (15, 1), padding=(7,  0), groups=C)
        ng = 8
        while ng > 1 and C % ng != 0:
            ng //= 2
        self.out_proj = nn.Conv2d(C * 5, channels, 1)
        self.out_norm = nn.GroupNorm(ng, channels)

    @staticmethod
    def _bidir(conv, x, dim):
        fwd = conv(x)
        rev = torch.flip(conv(torch.flip(x, [dim])), [dim])
        return 0.5 * (fwd + rev)

    def forward(self, x):
        proj     = self.in_proj(x)
        x_in, g  = proj.chunk(2, dim=1)
        x_in     = self.act(x_in)
        o1 = self._bidir(self.dw_h7,  x_in, 3)
        o2 = self._bidir(self.dw_v7,  x_in, 2)
        o3 = self._bidir(self.dw_h15, x_in, 3)
        o4 = self._bidir(self.dw_v15, x_in, 2)
        merged = torch.cat([o1, o2, o3, o4, x_in * self.act(g)], dim=1)
        return self.out_norm(self.out_proj(merged)) + x


class CFEB(nn.Module):
    """
    Conditional Feature Extraction Block.
    Conv(stride=ds) -> InstanceNorm -> ReLU -> Conv -> InstanceNorm -> ReLU
    """
    def __init__(self, in_channels, out_channels, downsample_factor=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,  out_channels, 3,
                               stride=downsample_factor, padding=1)
        self.in1   = nn.InstanceNorm2d(out_channels, affine=True)
        self.act1  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.in2   = nn.InstanceNorm2d(out_channels, affine=True)
        self.act2  = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.act1(self.in1(self.conv1(x)))
        x = self.act2(self.in2(self.conv2(x)))
        return x


class MCMFBlock(nn.Module):
    """
    Mamba Cross-Modal Fusion Module.
    Pipeline:
      1. Patch Embedding per modality: Conv -> LN
      2. Parallel SS2D (one per modality)
      3. Concat -> fuse_proj -> LN -> F_seq
      4. DF-FFN: Conv(1x1) -> GELU -> Dropout -> Conv(1x1) + residual
    """
    def __init__(self, channels, dropout=0.1):
        super().__init__()
        self.patch_embed_cloudy = nn.Conv2d(channels, channels, 3, padding=1)
        self.patch_embed_sar    = nn.Conv2d(channels, channels, 3, padding=1)
        self.ln_cloudy = nn.LayerNorm(channels)
        self.ln_sar    = nn.LayerNorm(channels)
        self.ss2d_cloudy = SS2D(channels)
        self.ss2d_sar    = SS2D(channels)
        self.fuse_proj   = nn.Conv2d(channels * 2, channels, 1)
        self.fuse_ln     = nn.LayerNorm(channels)
        expand = 4
        self.ffn = nn.Sequential(
            nn.Conv2d(channels, channels * expand, 1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(channels * expand, channels, 1),
        )

    @staticmethod
    def _apply_ln(x, ln):
        B, C, H, W = x.shape
        return (ln(x.permute(0, 2, 3, 1).reshape(B, H * W, C))
                  .reshape(B, H, W, C).permute(0, 3, 1, 2))

    def forward(self, f_cloudy, f_sar):
        p_c = self._apply_ln(self.patch_embed_cloudy(f_cloudy), self.ln_cloudy)
        p_s = self._apply_ln(self.patch_embed_sar(f_sar),       self.ln_sar)
        m_c = self.ss2d_cloudy(p_c)
        m_s = self.ss2d_sar(p_s)
        f_seq = self._apply_ln(self.fuse_proj(torch.cat([m_c, m_s], dim=1)), self.fuse_ln)
        return self.ffn(f_seq) + f_seq


class SAGF(nn.Module):
    """
    gate = Sigmoid(Conv(Concat(x, F_fused)))
    x'   = x*(1-gate) + F_fused*gate
    Returns (fused_feature, gate) so the gate can be used in the sparsity loss.
    """
    def __init__(self, channels):
        super().__init__()
        self.gate_conv = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x, f_fused):
        gate = self.gate_conv(torch.cat([x, f_fused], dim=1))
        return x * (1.0 - gate) + f_fused * gate, gate


class ResBlock(nn.Module):
    """
    GroupNorm -> SiLU -> Conv -> scale-shift(time) -> GroupNorm -> SiLU -> Conv
    + shortcut
    """
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=32):
        super().__init__()
        ng1 = num_groups
        while ng1 > 1 and in_ch % ng1 != 0:
            ng1 //= 2
        ng2 = num_groups
        while ng2 > 1 and out_ch % ng2 != 0:
            ng2 //= 2
        self.norm1    = nn.GroupNorm(ng1, in_ch)
        self.conv1    = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_emb_dim, out_ch * 2)   # scale + shift
        self.norm2    = nn.GroupNorm(ng2, out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.shortcut = (nn.Conv2d(in_ch, out_ch, 1)
                         if in_ch != out_ch else nn.Identity())

    def forward(self, x, t_emb):
        h            = self.conv1(F.silu(self.norm1(x)))
        t_out        = self.time_mlp(F.silu(t_emb))[:, :, None, None]
        scale, shift = t_out.chunk(2, dim=1)
        h            = h * (1.0 + scale) + shift
        h            = self.conv2(F.silu(self.norm2(h)))
        return h + self.shortcut(x)


class AttentionBlock(nn.Module):
    """Multi-head self-attention for global context at the lowest resolution."""
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = max(channels // num_heads, 1)
        self.scale     = self.head_dim ** -0.5
        ng = 32
        while ng > 1 and channels % ng != 0:
            ng //= 2
        self.norm = nn.GroupNorm(ng, channels)
        self.qkv  = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        q, k, v = qkv.unbind(dim=1)
        q = q.permute(0, 1, 3, 2)
        k = k.permute(0, 1, 3, 2)
        v = v.permute(0, 1, 3, 2)
        attn = F.softmax(torch.matmul(q, k.transpose(-2, -1)) * self.scale, dim=-1)
        out  = torch.matmul(attn, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.proj(out)


class OSMambaDiffUNet(nn.Module):
    def __init__(self, channels=128, time_emb_dim=512, mamba_channels=128):
        super().__init__()
        C, TC, MC = channels, time_emb_dim, mamba_channels

        # -- Conditional feature extraction --
        self.cfeb_cloudy = CFEB(3, MC, downsample_factor=4)
        self.cfeb_sar    = CFEB(2, MC, downsample_factor=4)
        self.mcmf        = MCMFBlock(MC)

        # -- Time embedding (sinusoidal -> MLP, Eq.9) --
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(C),
            nn.Linear(C, TC),
            nn.SiLU(),
            nn.Linear(TC, TC),
        )

        # -- Encoder --
        self.init_conv = nn.Conv2d(3, C, 3, padding=1)

        self.enc1  = nn.Sequential(ResBlock(C,     C,     TC), ResBlock(C,     C,     TC))
        self.down1 = nn.Conv2d(C,     C * 2, 3, stride=2, padding=1)

        self.enc2  = nn.Sequential(ResBlock(C * 2, C * 2, TC), ResBlock(C * 2, C * 2, TC))
        self.down2 = nn.Conv2d(C * 2, C * 4, 3, stride=2, padding=1)

        self.enc3  = nn.Sequential(ResBlock(C * 4, C * 4, TC), ResBlock(C * 4, C * 4, TC))
        self.down3 = nn.Conv2d(C * 4, C * 8, 3, stride=2, padding=1)

        self.enc4  = nn.Sequential(ResBlock(C * 8, C * 8, TC), ResBlock(C * 8, C * 8, TC))

        # -- SAGF lateral injections (after each DownSample, into encoder) --
        self.sagf2      = SAGF(C * 2)
        self.sagf3      = SAGF(C * 4)
        self.sagf4      = SAGF(C * 8)
        self.fuse_proj2 = nn.Conv2d(MC, C * 2, 1)
        self.fuse_proj3 = nn.Conv2d(MC, C * 4, 1)
        self.fuse_proj4 = nn.Conv2d(MC, C * 8, 1)

        # -- Bottleneck --
        self.mid1     = ResBlock(C * 8, C * 8, TC)
        self.mid_attn = AttentionBlock(C * 8, num_heads=8)
        self.mid2     = ResBlock(C * 8, C * 8, TC)

        # -- Decoder --
        self.up3  = nn.ConvTranspose2d(C * 8, C * 4, 2, stride=2)
        self.dec3 = nn.Sequential(ResBlock(C * 8, C * 4, TC), ResBlock(C * 4, C * 4, TC))

        self.up2  = nn.ConvTranspose2d(C * 4, C * 2, 2, stride=2)
        self.dec2 = nn.Sequential(ResBlock(C * 4, C * 2, TC), ResBlock(C * 2, C * 2, TC))

        self.up1  = nn.ConvTranspose2d(C * 2, C, 2, stride=2)
        self.dec1 = nn.Sequential(ResBlock(C * 2, C, TC), ResBlock(C, C, TC))

        # -- Output --
        ng = 32
        while ng > 1 and C % ng != 0:
            ng //= 2
        self.out_norm = nn.GroupNorm(ng, C)
        self.out_conv = nn.Conv2d(C, 3, 3, padding=1)

    @staticmethod
    def _run_seq(seq, x, t):
        for blk in seq:
            x = blk(x, t) if isinstance(blk, ResBlock) else blk(x)
        return x

    def forward(self, x_t, t, cloudy, sar):
        # 1. Condition encoding
        f_cloudy = self.cfeb_cloudy(cloudy)
        f_sar    = self.cfeb_sar(sar)
        f_fused  = self.mcmf(f_cloudy, f_sar)

        # 2. Time embedding
        t_emb = self.time_embed(t)

        # 3. Encoder
        x0 = self.init_conv(x_t)
        e1 = self._run_seq(self.enc1, x0, t_emb)

        x2 = self.down1(e1)
        e2 = self._run_seq(self.enc2, x2, t_emb)
        f2 = self.fuse_proj2(F.interpolate(f_fused, size=e2.shape[-2:],
                                           mode='bilinear', align_corners=False))
        e2_f, g2 = self.sagf2(e2, f2)

        x3 = self.down2(e2_f)
        e3 = self._run_seq(self.enc3, x3, t_emb)
        f3 = self.fuse_proj3(F.interpolate(f_fused, size=e3.shape[-2:],
                                           mode='bilinear', align_corners=False))
        e3_f, g3 = self.sagf3(e3, f3)

        x4 = self.down3(e3_f)
        e4 = self._run_seq(self.enc4, x4, t_emb)
        f4 = self.fuse_proj4(F.interpolate(f_fused, size=e4.shape[-2:],
                                           mode='bilinear', align_corners=False))
        e4_f, g4 = self.sagf4(e4, f4)

        # 4. Bottleneck
        mid = self.mid2(self.mid_attn(self.mid1(e4_f, t_emb)), t_emb)

        # 5. Decoder
        d3 = self._run_seq(self.dec3, torch.cat([self.up3(mid), e3_f], 1), t_emb)
        d2 = self._run_seq(self.dec2, torch.cat([self.up2(d3),  e2_f], 1), t_emb)
        d1 = self._run_seq(self.dec1, torch.cat([self.up1(d2),  e1],   1), t_emb)

        # 6. Output
        out = self.out_conv(F.silu(self.out_norm(d1)))
        return out, [g2, g3, g4]

# Smoke-test check
with torch.no_grad():
    _m = OSMambaDiffUNet(MODEL_CHANNELS, TIME_EMB_DIM, MAMBA_CHANNELS)
    _x = torch.randn(2, 3, 256, 256)
    _c = torch.randn(2, 3, 256, 256)
    _s = torch.randn(2, 2, 256, 256)
    _t = torch.randint(0, 100, (2,))
    _o, _g = _m(_x, _t, _c, _s)
    n = sum(p.numel() for p in _m.parameters())
    print(f'Model OK  output={_o.shape}  params={n/1e6:.2f}M  gates={len(_g)}')
del _m, _x, _c, _s, _t, _o, _g


In [ ]:
def cloud_loss(pred, clear, cloudy, m=0.2):
    """Differentiated constraint: penalise predictions too close to cloudy."""
    I_c = torch.mean(torch.abs(cloudy - clear), dim=1, keepdim=True)
    return torch.mean(I_c * F.relu(m - torch.abs(pred - cloudy)))

def gate_sparsity_loss(gates):
    """Push gate values toward 0 or 1."""
    return sum(torch.mean(g*(1-g)) for g in gates) / max(len(gates), 1)

def show_val_images(cloudy, clear, pred, epoch, n=4):
    """Display sample grid comparisons inline in the notebook without saving files."""
    try:
        from IPython.display import display, clear_output
        n = min(n, cloudy.size(0))
        fig, axes = plt.subplots(n, 3, figsize=(12, 3 * n))
        hdrs = ['Cloudy Input (LISS-IV)', 'Clear Ground Truth', 'OS-MambaDiff Prediction']
        for i in range(n):
            row = axes[i] if n > 1 else axes
            img_in = cloudy[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
            img_gt = clear[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
            img_pr = pred[i].cpu().permute(1, 2, 0).numpy().clip(0, 1)
            for j, (ax, img, hdr) in enumerate(zip(row, [img_in, img_gt, img_pr], hdrs)):
                ax.imshow(img)
                if i == 0:
                    ax.set_title(hdr, fontsize=10)
                ax.axis('off')
        plt.suptitle(f"Epoch {epoch} - Sample Visualizations", fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"  [vis] Could not show validation images: {e}")

class EMA:
    def __init__(self, model, decay=0.999):
        self.model = model; self.decay = decay
        self.shadow = {}; self.backup = {}
    def register(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad: self.shadow[n] = p.data.clone()
    def update(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n] = (1-self.decay)*p.data + self.decay*self.shadow[n]
    def apply_shadow(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone(); p.data.copy_(self.shadow[n])
    def restore(self):
        for n, p in self.model.named_parameters():
            if p.requires_grad: p.data.copy_(self.backup[n])
        self.backup = {}

def sigmoid_lr(epoch, total, lr_max, lr_min, a=10.0, b=0.5):
    p = epoch / total
    return lr_min + (lr_max-lr_min) / (1.0 + math.exp(a*(p-b)))

def save_ckpt(path, model, optimizer, ema, epoch, best_psnr):
    torch.save({
        'epoch': epoch, 'best_psnr': best_psnr,
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'ema_shadow': ema.shadow,
    }, path)
    print(f'  [ckpt] {path}')

print('Loss, EMA, LR, checkpoint & visualization utils ready.')


In [ ]:
# Datasets & Loaders
train_ds = LISS4Dataset(DATA_DIR, split='train')
val_ds   = LISS4Dataset(DATA_DIR, split='val')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS>0), drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS>0))
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

# Model
_base = OSMambaDiffUNet(
    channels       = MODEL_CHANNELS,
    time_emb_dim   = TIME_EMB_DIM,
    mamba_channels = MAMBA_CHANNELS
).to(device)

if COMPILE_MODEL:
    try:
        _base = torch.compile(_base)
        print('torch.compile() applied.')
    except Exception as e:
        print(f'[WARN] torch.compile() failed: {e}')
model = nn.DataParallel(_base) if num_gpus > 1 else _base

def raw_model(m):
    m = m.module if isinstance(m, nn.DataParallel) else m
    return getattr(m, '_orig_mod', m)

# Optimizer, EMA, Scheduler
optimizer = torch.optim.AdamW(raw_model(model).parameters(), lr=LR, weight_decay=0.0)
ema       = EMA(raw_model(model), decay=EMA_DECAY)
ema.register()
sched     = DiffusionScheduler(NUM_TIMESTEPS, NOISE_SCHEDULE)

print(f'Model params: {sum(p.numel() for p in raw_model(model).parameters())/1e6:.2f}M')

# AMP settings
scaler       = torch.cuda.amp.GradScaler(enabled=(MIXED_PRECISION == 'fp16'))
_AMP_ENABLED = MIXED_PRECISION != 'none'
_AMP_DTYPE   = {'fp16': torch.float16, 'bf16': torch.bfloat16}.get(MIXED_PRECISION, torch.float32)

# Training state — ALWAYS auto-resume if resume.pth exists
start_epoch  = 1
best_psnr    = -1.0
es_counter   = 0
es_best_psnr = -1.0
history = {
    'epoch': [],
    'train_loss':[], 'train_mse':[], 'train_ssim_loss':[],
    'train_cloud':[], 'train_gate':[],
    'val_loss':[], 'val_psnr':[], 'val_ssim':[], 'val_sam':[],
}

resume_path = '/kaggle/input/models/bro192007/final/pytorch/default/1/best_model (1).pth'
if os.path.isfile(resume_path):
    try:
        ckpt = torch.load(resume_path, map_location=device)
        raw_model(model).load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        ema.shadow = ckpt['ema_shadow']
        start_epoch = ckpt['epoch'] + 1
        best_psnr = ckpt['best_psnr']
        es_best_psnr = best_psnr
        print(f"✓ Resumed training from epoch {ckpt['epoch']} | best_psnr = {best_psnr:.4f} dB")
    except Exception as e:
        print(f"⚠ Could not resume from checkpoint: {e}. Starting from epoch 1.")
else:
    print("No resume.pth checkpoint found. Starting from scratch (epoch 1).")


In [ ]:
print(f'Training: {EPOCHS} epochs | batch={BATCH_SIZE} | BF16={MIXED_PRECISION} | compile={COMPILE_MODEL}')
print('='*80)

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    # LR
    lr_now = sigmoid_lr(epoch, EPOCHS, LR, LR_MIN)
    for pg in optimizer.param_groups: pg['lr'] = lr_now

    # ── TRAIN ────────────────────────────────────────────────────────────
    model.train()
    s_loss = s_mse = s_ssim = s_cloud = s_gate = 0.0
    n = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:04d}/{EPOCHS}',
                leave=True, dynamic_ncols=True)
    for batch in pbar:
        clear  = batch['clear'].to(device, non_blocking=True)
        cloudy = batch['cloudy'].to(device, non_blocking=True)
        sar    = batch['sar'].to(device, non_blocking=True)
        B      = clear.size(0)
        t      = torch.randint(0, NUM_TIMESTEPS, (B,), device=device).long()

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=_AMP_ENABLED, dtype=_AMP_DTYPE):
            xt       = sched.add_noise(clear, t)
            pred, gates = model(xt, t, cloudy, sar)
            L_mse   = F.mse_loss(pred, clear)
            L_ssim  = 1.0 - compute_ssim(pred, clear)
            L_cloud = cloud_loss(pred, clear, cloudy, CLOUD_MARGIN)
            L_gate  = gate_sparsity_loss(gates)
            loss    = (LAMBDA_MSE*L_mse + LAMBDA_SSIM*L_ssim
                     + LAMBDA_CLOUD*L_cloud + LAMBDA_GATE*L_gate)

        if MIXED_PRECISION == 'fp16':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(raw_model(model).parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(raw_model(model).parameters(), GRAD_CLIP)
            optimizer.step()

        ema.update()
        n += B
        s_loss  += loss.item()    * B
        s_mse   += L_mse.item()  * B
        s_ssim  += L_ssim.item() * B
        s_cloud += L_cloud.item()* B
        s_gate  += L_gate.item() * B

        pbar.set_postfix(
            loss =f'{s_loss/n:.4f}',
            mse  =f'{s_mse/n:.5f}',
            ssim =f'{1-s_ssim/n:.4f}',
            lr   =f'{lr_now:.1e}',
        )

    e_loss  = s_loss/n; e_mse  = s_mse/n; e_ssim = s_ssim/n
    e_cloud= s_cloud/n; e_gate = s_gate/n
    elapsed = time.time()-t0

    print(f'Epoch {epoch:04d}/{EPOCHS} | '
          f'loss={e_loss:.4f}  mse={e_mse:.5f}  ssim_l={e_ssim:.4f}  '
          f'cloud={e_cloud:.4f}  gate={e_gate:.5f}  '
          f'lr={lr_now:.2e}  {elapsed:.1f}s')

    # ── PERIODIC CHECKPOINT ───────────────────────────────────────────────
    if epoch % SAVE_INTERVAL == 0:
        ema.apply_shadow()
        save_ckpt(os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch:04d}.pth'),
                  raw_model(model), optimizer, ema, epoch, best_psnr)
        ema.restore()

    # ── VALIDATION ────────────────────────────────────────────────────────
    if epoch % VAL_INTERVAL == 0 or epoch == EPOCHS:
        ema.apply_shadow()
        model.eval()

        vl = vp = vs = vsa = 0.0
        vn = 0

        vis_cloudy = vis_clear = vis_pred = None

        with torch.no_grad():
            for vi, vbatch in enumerate(
                tqdm(val_loader, desc=f'  VAL {epoch:04d}',
                     leave=False, dynamic_ncols=True)
            ):
                vc     = vbatch['clear'].to(device)
                vcloud = vbatch['cloudy'].to(device)
                vsar   = vbatch['sar'].to(device)
                vB     = vc.size(0)

                # Val loss (at random t — fast)
                vt = torch.randint(0, NUM_TIMESTEPS, (vB,), device=device).long()
                with torch.cuda.amp.autocast(enabled=_AMP_ENABLED, dtype=_AMP_DTYPE):
                    vxt         = sched.add_noise(vc, vt)
                    vp0l, vgates= model(vxt, vt, vcloud, vsar)
                    vL_mse  = F.mse_loss(vp0l, vc)
                    vL_ssim = 1.0 - compute_ssim(vp0l, vc)
                    vL_cloud= cloud_loss(vp0l, vc, vcloud, CLOUD_MARGIN)
                    vL_gate = gate_sparsity_loss(vgates)
                    vloss   = (LAMBDA_MSE*vL_mse + LAMBDA_SSIM*vL_ssim
                             + LAMBDA_CLOUD*vL_cloud + LAMBDA_GATE*vL_gate)
                vl += vloss.item() * vB

                # DDIM sampling -> PSNR / SSIM / SAM
                with torch.cuda.amp.autocast(enabled=_AMP_ENABLED, dtype=_AMP_DTYPE):
                    vpred = sched.sample_ddim(model, vcloud, vsar,
                                             steps=DDIM_STEPS, eta=DDIM_ETA)
                vpred = vpred.float()
                vp  += compute_psnr(vc, vpred).item() * vB
                vs  += compute_ssim(vc, vpred).item() * vB
                vsa += compute_sam (vc, vpred).item() * vB
                vn  += vB

                if vis_cloudy is None:
                    vis_cloudy = vcloud.cpu()
                    vis_clear  = vc.cpu()
                    vis_pred   = vpred.cpu()

                if MAX_VAL_BATCHES > 0 and vi+1 >= MAX_VAL_BATCHES:
                    break

        ema.restore()
        save_ckpt(os.path.join(CHECKPOINT_DIR, 'resume.pth'), raw_model(model), optimizer, ema, epoch, best_psnr)

        mv_loss = vl/vn
        mv_psnr = vp/vn
        mv_ssim = vs/vn
        mv_sam  = vsa/vn

        print(f'  --> VAL | val_loss={mv_loss:.4f}  '
              f'PSNR={mv_psnr:.3f}dB  SSIM={mv_ssim:.4f}  '
              f'SAM={mv_sam:.4f}rad ({math.degrees(mv_sam):.2f}deg)')

        # Show validation images inline every vis_interval epochs
        if epoch % VIS_INTERVAL == 0 or epoch == EPOCHS:
            if vis_cloudy is not None:
                show_val_images(vis_cloudy, vis_clear, vis_pred, epoch, n=VIS_SAMPLES)

        # History
        history['epoch'].append(epoch)
        history['train_loss'].append(e_loss); history['train_mse'].append(e_mse)
        history['train_ssim_loss'].append(e_ssim)
        history['train_cloud'].append(e_cloud); history['train_gate'].append(e_gate)
        history['val_loss'].append(mv_loss); history['val_psnr'].append(mv_psnr)
        history['val_ssim'].append(mv_ssim); history['val_sam'].append(mv_sam)

        # Best model
        if mv_psnr > best_psnr:
            best_psnr = mv_psnr
            ema.apply_shadow()
            save_ckpt(os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
                      raw_model(model), optimizer, ema, epoch, best_psnr)
            ema.restore()
            print(f'  BEST PSNR = {best_psnr:.3f} dB')

        # Early stopping
        if mv_psnr > es_best_psnr + ES_MIN_DELTA:
            es_best_psnr = mv_psnr; es_counter = 0
        else:
            es_counter += 1
            print(f'  [ES] {es_counter}/{ES_PATIENCE} rounds no improvement '
                  f'(best={es_best_psnr:.3f}dB)')
            if es_counter >= ES_PATIENCE:
                print(f'  [ES] Triggered at epoch {epoch}. Best PSNR={es_best_psnr:.3f}dB')
                break

    print('─'*80)

print(f'Training complete. Best val PSNR = {best_psnr:.3f} dB')


In [ ]:
if not history['epoch']:
    print('No history to plot.')
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('OS-MambaDiff — Training Curves', fontsize=14, fontweight='bold')
    ep = history['epoch']
    specs = [
        (axes[0,0], history['train_loss'],    'Train Loss',     'royalblue'),
        (axes[0,1], history['val_loss'],      'Val Loss',       'darkorange'),
        (axes[0,2], history['val_psnr'],      'Val PSNR (dB)', 'seagreen'),
        (axes[1,0], history['val_ssim'],      'Val SSIM',       'mediumorchid'),
        (axes[1,1], history['val_sam'],       'Val SAM (rad)', 'tomato'),
        (axes[1,2], history['train_mse'],     'Train MSE',      'cadetblue'),
    ]
    for ax, y, ylabel, color in specs:
        ax.plot(ep, y, color=color, linewidth=1.2)
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(ylabel); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(CHECKPOINT_DIR, 'training_curves.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved -> {p}')
    with open(os.path.join(CHECKPOINT_DIR, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    print(f'Saved -> {os.path.join(CHECKPOINT_DIR, "history.json")}')


In [ ]:
import os, time
import rasterio
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
matched_filename = 'pair10_001504.tif'
best_path = '/kaggle/input/models/bro192007/final/pytorch/default/1/best_model (1).pth'

# ============================================================
# 1. Paths
# ============================================================
cloudy_path = os.path.join(DATA_DIR, 'cloudy', matched_filename)
sar_path    = os.path.join(DATA_DIR, 'sar',    matched_filename)
clear_path  = os.path.join(DATA_DIR, 'clear',  matched_filename)

# ============================================================
# 2. Load + normalize (matches LISS4Dataset preprocessing exactly)
# ============================================================
with rasterio.open(cloudy_path) as src:
    cloudy_arr = src.read()  # (3, H, W) uint16
with rasterio.open(sar_path) as src:
    sar_arr = src.read()     # (2, H, W) float64 dB
with rasterio.open(clear_path) as src:
    clear_arr = src.read()   # (3, H, W) uint16

cloudy_norm = normalize_optical(cloudy_arr)
sar_norm    = normalize_sar(sar_arr)
clear_norm  = normalize_optical(clear_arr)

cloudy_t = torch.from_numpy(cloudy_norm).unsqueeze(0).float().to(device)
sar_t    = torch.from_numpy(sar_norm).unsqueeze(0).float().to(device)
clear_t  = torch.from_numpy(clear_norm).unsqueeze(0).float().to(device)

# ============================================================
# 3. Load checkpoint
# ============================================================
ckpt = torch.load(best_path, map_location=device)
raw_model(model).load_state_dict(ckpt['model'])
ema.shadow = ckpt['ema_shadow']
print(f'Loaded checkpoint: epoch={ckpt["epoch"]}  best_psnr={ckpt["best_psnr"]:.3f}dB')

ema.apply_shadow()
model.eval()

# ============================================================
# 4. Warm-up run (discard — avoids torch.compile() overhead skewing timing)
# ============================================================
with torch.no_grad(), torch.amp.autocast('cuda', enabled=_AMP_ENABLED, dtype=_AMP_DTYPE):
    _ = sched.sample_ddim(model, cloudy_t, sar_t, steps=DDIM_STEPS, eta=0.0)
torch.cuda.synchronize()

# ============================================================
# 5. Timed inference (CUDA events = GPU-accurate timing)
# ============================================================
starter = torch.cuda.Event(enable_timing=True)
ender   = torch.cuda.Event(enable_timing=True)

starter.record()
with torch.no_grad(), torch.amp.autocast('cuda', enabled=_AMP_ENABLED, dtype=_AMP_DTYPE):
    pred = sched.sample_ddim(model, cloudy_t, sar_t, steps=DDIM_STEPS, eta=0.0)
ender.record()
torch.cuda.synchronize()

infer_time_ms = starter.elapsed_time(ender)
pred = pred.float()

print(f'\n--- Inference: {matched_filename} ---')
print(f'Inference time: {infer_time_ms:.2f} ms  ({infer_time_ms/1000:.4f} s)  '
      f'[{DDIM_STEPS} DDIM steps -> {infer_time_ms/DDIM_STEPS:.2f} ms/step]')
print(f'Peak GPU mem  : {torch.cuda.max_memory_allocated(device)/1e9:.2f} GB')

# ============================================================
# 6. Metrics
# ============================================================
mse_val  = F.mse_loss(pred, clear_t).item()
psnr_val = compute_psnr(clear_t, pred).item()
ssim_val = compute_ssim(clear_t, pred).item()
sam_val  = compute_sam(clear_t, pred).item()

print(f'MSE : {mse_val:.6f}')
print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')
print(f'SAM : {sam_val:.4f} rad')

# ============================================================
# 7. Visualize
# ============================================================
cloudy_img = cloudy_t[0].cpu().permute(1, 2, 0).numpy().clip(0, 1)
clear_img  = clear_t[0].cpu().permute(1, 2, 0).numpy().clip(0, 1)
pred_img   = pred[0].cpu().permute(1, 2, 0).numpy().clip(0, 1)

imgs = [cloudy_img, clear_img, pred_img]
hdrs = ['Cloudy (LISS-IV)', 'Clear GT',
        f'DDIM-{DDIM_STEPS} Pred\nPSNR={psnr_val:.2f}dB  MSE={mse_val:.5f}']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, hdr in zip(axes, imgs, hdrs):
    ax.imshow(img)
    ax.set_title(hdr, fontsize=10)
    ax.axis('off')

plt.tight_layout()
out_path = os.path.join(CHECKPOINT_DIR, f'{matched_filename.replace(".tif","")}_inference.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved -> {out_path}')

ema.restore()